# Hafta 6 — Yinelemeli Mimariler: RNN, LSTM, GRU

Temel ilkelerden sıralı modeller. Zaman içinde geri yayılımı (BPTT) türetiyor, kaybolan gradyan problemini analitik olarak teşhis ediyor ve LSTM ile GRU hücrelerini elle gradyan hesaplayarak sıfırdan kuruyoruz. Uygulamalarımızı kayan nokta hassasiyetinde sayısal gradyanlarla doğruluyoruz.

## Öğrenme Hedefleri

- Basit bir RNN için BPTT'yi elle türetmek ve kaybolan gradyanların kaynağını belirlemek.
- Kaybolan gradyan teoremini ifade etmek ve göstermek (Bengio, Simard, Frasconi, 1994).
- Bir LSTM hücresini elle türetilmiş gradyanlarla NumPy'da sıfırdan uygulamak.
- Shakespeare üzerinde karakter düzeyinde bir RNN eğitmek ve örnekler üretmek.

## Zorunlu Okumalar

- Hochreiter, S. ve Schmidhuber, J. (1997). *Long Short-Term Memory*.
- Bengio, Y., Simard, P. ve Frasconi, P. (1994). *Learning Long-Term Dependencies with Gradient Descent Is Difficult*.
- Cho, K. vd. (2014). *Learning Phrase Representations using RNN Encoder–Decoder*.

In [ ]:
import sys, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    torch.manual_seed(0)
except ImportError:
    HAS_TORCH = False

## 1. Basit RNN

$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b}).$$

$t$ anındaki bir $\mathcal{L}_t$ kaybı için $t - k$ anına geri akan gradyan $k$ Jacobian'ın çarpımını içerir:

$$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-k}} = \prod_{i=t-k+1}^{t} \frac{\partial \mathbf{h}_i}{\partial \mathbf{h}_{i-1}} = \prod_{i} \text{diag}(1 - \mathbf{h}_i^2) \cdot W_h.$$

$W_h$'nin spektral yarıçapı 1'den küçükse (ve tanh türevleri 1 ile sınırlı olduğundan) bu çarpım $k$ ile üstel olarak kaybolur. 1'den büyükse patlayabilir.

In [ ]:
class SimpleRNN:
    def __init__(self, in_dim, hidden_dim, scale=0.1):
        self.Wh = np.random.randn(hidden_dim, hidden_dim) * scale
        self.Wx = np.random.randn(hidden_dim, in_dim) * scale
        self.b  = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, xs):
        T = len(xs)
        hs = [np.zeros(self.hidden_dim)]
        for t in range(T):
            h_new = np.tanh(self.Wh @ hs[-1] + self.Wx @ xs[t] + self.b)
            hs.append(h_new)
        return hs[1:]  # uzunluk T

# Kaybolan/patlayan rejimleri deneysel olarak göster.
def gradient_norm_through_time(scale, T=50, trials=20):
    norms = []
    for _ in range(trials):
        rnn = SimpleRNN(in_dim=8, hidden_dim=32, scale=scale)
        xs = [np.random.randn(8) for _ in range(T)]
        hs = rnn.forward(xs)
        # ∂h_T / ∂h_0 = ∏ diag(1 - h_i^2) W_h
        J = np.eye(32)
        for t in range(T):
            J = (np.diag(1 - hs[t]**2) @ rnn.Wh) @ J if t > 0 else (np.diag(1 - hs[t]**2) @ rnn.Wh)
        norms.append(np.linalg.norm(J))
    return np.mean(norms)

scales = [0.05, 0.1, 0.3, 0.5, 1.0, 1.5]
norms_T = []
for s in scales:
    g = gradient_norm_through_time(scale=s, T=30)
    norms_T.append(g)
    print(f"başlatma ölçeği = {s:.2f}  ->  ||∂h_30 / ∂h_0|| ≈ {g:.2e}")

plt.figure(figsize=(8, 4))
plt.semilogy(scales, norms_T, marker='o')
plt.xlabel('ağırlık başlatma ölçeği'); plt.ylabel('gradyan normu (log)')
plt.title('Basit bir RNN\'de kaybolan / patlayan gradyanlar (T=30)')
plt.grid(True, which='both', alpha=0.3); plt.tight_layout(); plt.show()

## 2. LSTM hücresi

Dört kapı, hücre durumu $\mathbf{c}_t$'yi ve gizli durum $\mathbf{h}_t$'yi yönetir:

$$
\begin{aligned}
\mathbf{f}_t &= \sigma(W_f [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_f) & \text{unutma kapısı}\\
\mathbf{i}_t &= \sigma(W_i [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_i) & \text{giriş kapısı}\\
\tilde{\mathbf{c}}_t &= \tanh(W_c [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_c) & \text{aday}\\
\mathbf{o}_t &= \sigma(W_o [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_o) & \text{çıkış kapısı}\\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \\
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t)
\end{aligned}
$$

Temel içgörü: $\mathbf{c}_t$ *toplamsal* olarak güncellenir. $\partial \mathbf{c}_t / \partial \mathbf{c}_{t-1}$'nin doğrudan yolu $\mathbf{f}_t$'dir (köşegen); unutma kapısı 1'e yakın kaldığı sürece gradyanlar tekrarlanan matris çarpımlarıyla bileşikleşmeden zaman içinde akar. Bu, LSTM'in kaybolan gradyan problemine getirdiği yapısal çözümdür. Unutma kapısı sapmasını 1 ile başlatmak (Jozefowicz vd., 2015) kapının eğitimin başından itibaren açık kalmasını sağlar — bunun etkisini 4. bölümdeki deneyde göreceğiz.

In [ ]:
class LSTMCell:
    def __init__(self, in_dim, hidden_dim, scale=0.1):
        D = in_dim + hidden_dim
        H = hidden_dim
        self.W = np.random.randn(4 * H, D) * scale     # üst üste yığılmış W_f, W_i, W_c, W_o
        self.b = np.zeros(4 * H)
        # Unutma kapısı sapmasını 1 ile başlat (Jozefowicz vd., 2015) — eğitimi hızlandırır.
        self.b[:H] = 1.0
        self.H, self.D = H, D

    def forward(self, x, h_prev, c_prev):
        z = np.concatenate([h_prev, x])
        a = self.W @ z + self.b  # (4H,)
        H = self.H
        f = 1 / (1 + np.exp(-a[:H]))
        i = 1 / (1 + np.exp(-a[H:2*H]))
        g = np.tanh(a[2*H:3*H])
        o = 1 / (1 + np.exp(-a[3*H:]))
        c = f * c_prev + i * g
        h = o * np.tanh(c)
        cache = (x, h_prev, c_prev, f, i, g, o, c)
        return h, c, cache

    def backward(self, dh, dc, cache):
        x, h_prev, c_prev, f, i, g, o, c = cache
        H = self.H
        # h = o * tanh(c)
        do = dh * np.tanh(c)
        dc = dc + dh * o * (1 - np.tanh(c) ** 2)
        df = dc * c_prev
        di = dc * g
        dg = dc * i
        dc_prev = dc * f

        # Kapı ön-aktivasyonları:  σ' = σ(1-σ),  tanh' = 1 - tanh^2.
        df_a = df * f * (1 - f)
        di_a = di * i * (1 - i)
        dg_a = dg * (1 - g * g)
        do_a = do * o * (1 - o)
        da = np.concatenate([df_a, di_a, dg_a, do_a])

        z = np.concatenate([h_prev, x])
        dW = np.outer(da, z)
        db = da
        dz = self.W.T @ da
        dh_prev = dz[:H]
        dx = dz[H:]
        return dx, dh_prev, dc_prev, dW, db

# Analitik gradyana karşı sayısal gradyan denetimi
def numerical_grad(cell, x, h_prev, c_prev, dh, dc, eps=1e-6):
    # L = dh·h + dc·c olmak üzere ∂L / ∂W'yi denetle.
    n_grad = np.zeros_like(cell.W)
    for idx in np.ndindex(cell.W.shape):
        cell.W[idx] += eps
        h_p, c_p, _ = cell.forward(x, h_prev, c_prev)
        L_p = dh @ h_p + dc @ c_p
        cell.W[idx] -= 2 * eps
        h_m, c_m, _ = cell.forward(x, h_prev, c_prev)
        L_m = dh @ h_m + dc @ c_m
        cell.W[idx] += eps
        n_grad[idx] = (L_p - L_m) / (2 * eps)
    return n_grad

cell = LSTMCell(in_dim=4, hidden_dim=3, scale=0.5)
x = np.random.randn(4); h0 = np.random.randn(3) * 0.1; c0 = np.random.randn(3) * 0.1
dh_out = np.random.randn(3); dc_out = np.random.randn(3)

h1, c1, cache = cell.forward(x, h0, c0)
dx, dh_prev, dc_prev, dW_analytic, db_analytic = cell.backward(dh_out, dc_out, cache)
dW_numeric = numerical_grad(cell, x, h0, c0, dh_out, dc_out)

err = np.max(np.abs(dW_analytic - dW_numeric))
print(f"dW için max |analitik - sayısal| = {err:.2e}")
print("(< 1e-6 olmalı — elle türetilen gradyanları doğrular.)")

## 3. Shakespeare üzerinde karakter düzeyinde LSTM

Karpathy'nin klasik kurulumu: bir önek verildiğinde sonraki karakteri tahmin et. Küçücük bir LSTM bile yazımı, noktalamayı ve sahte Elizabeth dönemi ritmini yakalar. Burada hız için PyTorch kullanıyoruz, ama hücre matematiği tam olarak yukarıda kurduğumuz şeydir.

> Not: Aşağıdaki derlem yalnızca birkaç dizenin tekrarından oluşur; model bu dizeleri büyük ölçüde ezberler. Gerçek bir deney için tam *tinyshakespeare* metnini kullanın.

In [ ]:
if HAS_TORCH:
    # Shakespeare tadında küçük bir derlem. Gerçek eğitim için tam tinyshakespeare ile değiştirin.
    TEXT = ("ROMEO: Tut, I have lost myself; I am not here; "
            "This is not Romeo, he's some other where. "
            "JULIET: O Romeo, Romeo! wherefore art thou Romeo? "
            "Deny thy father and refuse thy name; "
            "Or, if thou wilt not, be but sworn my love, "
            "And I'll no longer be a Capulet. " * 30)

    chars = sorted(set(TEXT))
    ch2id = {c: i for i, c in enumerate(chars)}
    id2ch = {i: c for c, i in ch2id.items()}
    V = len(chars)

    data = torch.tensor([ch2id[c] for c in TEXT])

    class CharLSTM(nn.Module):
        def __init__(self, V, emb=16, hidden=64):
            super().__init__()
            self.emb = nn.Embedding(V, emb)
            self.lstm = nn.LSTM(emb, hidden, batch_first=True)
            self.head = nn.Linear(hidden, V)

        def forward(self, x, state=None):
            e = self.emb(x)
            out, state = self.lstm(e, state)
            return self.head(out), state

    model = CharLSTM(V, emb=16, hidden=64)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)

    SEQ = 50; BATCH = 16
    def get_batch():
        ix = torch.randint(0, len(data) - SEQ - 1, (BATCH,))
        x = torch.stack([data[i:i+SEQ] for i in ix])
        y = torch.stack([data[i+1:i+SEQ+1] for i in ix])
        return x, y

    for step in range(300):
        x, y = get_batch()
        logits, _ = model(x)
        loss = nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % 100 == 0:
            print(f"adım {step+1:4d}  kayıp = {loss.item():.3f}")

    # Örnek üret
    model.eval()
    prompt = "ROMEO:"
    state = None
    inp = torch.tensor([[ch2id[c] for c in prompt]])
    with torch.no_grad():
        for _ in range(120):
            logits, state = model(inp, state)
            probs = nn.functional.softmax(logits[0, -1] / 0.8, dim=-1)
            nxt = torch.multinomial(probs, 1)
            prompt += id2ch[nxt.item()]
            inp = nxt.view(1, 1)
    print("\nÖrnek:\n", prompt)
else:
    print("Karakter düzeyinde LSTM bölümünü çalıştırmak için PyTorch kurun.")

## 4. GRU — daha basit bir kapılı varyant

Cho vd. (2014) unutma ve giriş kapılarını tek bir güncelleme kapısında birleştirdi:

$$
\begin{aligned}
\mathbf{z}_t &= \sigma(W_z [\mathbf{h}_{t-1}; \mathbf{x}_t]) & \text{güncelleme kapısı} \\
\mathbf{r}_t &= \sigma(W_r [\mathbf{h}_{t-1}; \mathbf{x}_t]) & \text{sıfırlama kapısı} \\
\tilde{\mathbf{h}}_t &= \tanh(W [\mathbf{r}_t \odot \mathbf{h}_{t-1}; \mathbf{x}_t]) \\
\mathbf{h}_t &= (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t
\end{aligned}
$$

LSTM'den daha az parametre, benzer deneysel performans (Jozefowicz vd., 2015 — açık bir kazanan yok).

Aşağıdaki deneyde üç hücre türünü uzun menzilli bir hatırlama görevinde karşılaştırıyoruz: model 20 tokenlık bir diziyi okur ve sonunda **ilk** tokenı söylemelidir. 9 olası değer olduğundan şans düzeyi ≈ 0,11'dir. LSTM'de unutma kapısı sapması 1 ile başlatılır (2. bölüm); bu yapılmazsa LSTM bazı tohumlarda başlangıç platosundan çıkamaz.

In [ ]:
# Oyuncak bir uzun menzilli görevde PyTorch yerleşikleriyle hızlı bir GRU / LSTM / RNN karşılaştırması.
if HAS_TORCH:
    # Görev: ilk tokenı hatırla, T adım sonra onu çıktı olarak ver.
    def make_recall_data(N=200, T=15, V=8):
        x = torch.randint(1, V, (N, T))
        y = x[:, 0].clone()
        return x, y

    class Recurrent(nn.Module):
        def __init__(self, V, cell_type, hidden=32):
            super().__init__()
            self.emb = nn.Embedding(V, hidden)
            self.rnn = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[cell_type](
                hidden, hidden, batch_first=True)
            self.head = nn.Linear(hidden, V)
            if cell_type == 'lstm':
                # Unutma kapısı sapması = 1 (Jozefowicz vd., 2015). PyTorch kapı sırası: i, f, g, o.
                with torch.no_grad():
                    for name, p in self.rnn.named_parameters():
                        if name.startswith('bias'):
                            p[hidden:2 * hidden].fill_(1.0 if name.startswith('bias_ih') else 0.0)
        def forward(self, x):
            e = self.emb(x); o, _ = self.rnn(e); return self.head(o[:, -1])

    torch.manual_seed(0)
    Xtr, ytr = make_recall_data(N=2048, T=20, V=10)
    Xte, yte = make_recall_data(N=512, T=20, V=10)

    EPOCHS, BATCH = 45, 64
    results = {}
    for cell_type in ['rnn', 'lstm', 'gru']:
        torch.manual_seed(0)
        m = Recurrent(10, cell_type, hidden=32)
        opt = torch.optim.Adam(m.parameters(), lr=5e-3)
        for ep in range(EPOCHS):
            perm = torch.randperm(len(Xtr))
            for k in range(0, len(Xtr), BATCH):
                idx = perm[k:k + BATCH]
                opt.zero_grad()
                loss = nn.functional.cross_entropy(m(Xtr[idx]), ytr[idx])
                loss.backward(); opt.step()
        with torch.no_grad():
            acc = (m(Xte).argmax(-1) == yte).float().mean().item()
        results[cell_type] = acc
        print(f"{cell_type:>5}  {EPOCHS} dönemden sonra test doğruluğu: {acc:.3f}")
    print("\nBeklenen: sade RNN şans düzeyinde (~0.11) kalır; LSTM ve GRU bu 20 adımlık hatırlama görevini çözer.")
else:
    print("Atlandı (PyTorch yok).")

## 5. Alıştırmalar

1. **Kaybolan gradyan teoremini kanıtlayın.** $\sigma$ aktivasyonlu ve spektral yarıçapı $\rho(W_h) < 1$ olan basit bir RNN için, $\partial \mathbf{h}_t / \partial \mathbf{h}_{t-k}$ gradyanının operatör normunun bir $C$ sabiti için $C \rho^k$ ile üstten sınırlı olduğunu gösterin. Uzun menzilli öğrenme sinyalinin üstel olarak kaybolduğu sonucuna varın.
2. **Gradyan kırpma.** $\tau$ eşikli gradyan kırpmayı uygulayın. Patlayan gradyan görevinde eğitim kararsızlığını ortadan kaldırdığını, kaybolan rejimi ise etkilemediğini deneysel olarak gösterin.
3. **Çift yönlü LSTM.** Ters yönlerde iki LSTM'i üst üste koyup birleştirin. Sözcük türü (POS) etiketleme üzerinde eğitin ve tek yönlü bir modelle karşılaştırın.
4. **LSTM'de katman normalizasyonu.** Kapıların içine katman normalizasyonu ekleyin (Ba, Kiros, Hinton, 2016). Eğitim kararlılığına ve nihai perpleksiteye etkisini ölçün.
5. **Unutma kapısı ablasyonu.** 4. bölümdeki deneyi unutma kapısı sapması 0 ile (PyTorch varsayılanı) birkaç farklı tohumla tekrarlayın. LSTM ne sıklıkla platoda kalıyor?

---

## Gelecek Hafta

Hafta 7 — Diziden diziye modeller ve dikkat. Kodlayıcı–kod çözücü paradigması ve onun darboğazını kıran dikkat mekanizması.